In [2]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [3]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [ ]:

queryPerson = """
SELECT 
  [BusinessEntityID],
  [NameStyle],
  [Title],
  [FirstName],
  [MiddleName],
  [LastName],
  [Suffix]
FROM Person.Person
"""
tablaPerson = pd.read_sql_query(queryPerson, motorBaseDatos)

queryPersonPhone = """
SELECT 
  [BusinessEntityID],
  [PhoneNumber]
FROM Person.PersonPhone
"""
tablaPersonPhone = pd.read_sql_query(queryPersonPhone, motorBaseDatos)

queryEmailAddress = """
SELECT 
  [BusinessEntityID],
  [EmailAddress]
FROM Person.EmailAddress
"""
tablaEmailAddress = pd.read_sql_query(queryEmailAddress, motorBaseDatos)

query = """
SELECT 
  [AddressID],
  [AddressLine1],
  [AddressLine2],
  [StateProvinceID]
FROM Person.Address
"""
tablaAddress = pd.read_sql_query(query, motorBaseDatos)

queryBusinessEntityAddress = """
SELECT 
  [BusinessEntityID],
  [AddressID]
FROM Person.BusinessEntityAddress
"""
tablaBusinessEntityAddress = pd.read_sql_query(queryBusinessEntityAddress, motorBaseDatos)

queryStateProvince = """
SELECT 
[StateProvinceID]
      ,[StateProvinceCode]
FROM Person.StateProvince
"""
tablaStateProvince = pd.read_sql_query(queryStateProvince, motorBaseDatos)


queryEmployee = """
SELECT 
  [BusinessEntityID],
  [BirthDate],
  [MaritalStatus],
  [Gender]
FROM HumanResources.Employee
"""
tablaEmployee = pd.read_sql_query(queryEmployee, motorBaseDatos)




queryCustomer = """
SELECT 
  [CustomerID],
  [AccountNumber]
FROM Sales.Customer
"""
tablaCustomer = pd.read_sql_query(queryCustomer, motorBaseDatos)









In [21]:
customer = tablaBusinessEntityAddress.merge(tablaCustomer, left_on='BusinessEntityID', right_on='CustomerID')

customer.drop(columns=[
    'AccountNumber',
    'CustomerID'
],inplace=True)
customer

,BusinessEntityID,AddressID
0,1,249
1,2,293
2,3,224
3,4,11387
4,5,190
...,...,...
10285,20099,206
10286,20305,217
10287,20419,222
10288,20550,255


TRANSFORMACION

In [22]:
customer = customer.merge(tablaAddress, on='AddressID')
customer

,BusinessEntityID,AddressID,AddressLine1,AddressLine2,StateProvinceID
0,1,249,4350 Minute Dr.,None,79
1,2,293,7559 Worth Ct.,None,79
2,3,224,2137 Birchwood Dr,None,79
3,4,11387,5678 Lakeview Blvd.,None,36
4,5,190,9435 Breck Court,None,79
...,...,...,...,...,...
10285,20099,206,6097 Mt. McKinley Ct.,None,79
10286,20305,217,1960 Via Catanzaro,None,79
10287,20419,222,7723 Firestone Drive,None,79
10288,20550,255,7469 Paradise Ct.,None,79


In [23]:
customer = customer.merge(tablaStateProvince, on='StateProvinceID')
customer

,BusinessEntityID,AddressID,AddressLine1,AddressLine2,StateProvinceID,StateProvinceCode
0,1,249,4350 Minute Dr.,None,79,WA
1,2,293,7559 Worth Ct.,None,79,WA
2,3,224,2137 Birchwood Dr,None,79,WA
3,4,11387,5678 Lakeview Blvd.,None,36,MN
4,5,190,9435 Breck Court,None,79,WA
...,...,...,...,...,...,...
10285,20099,206,6097 Mt. McKinley Ct.,None,79,WA
10286,20305,217,1960 Via Catanzaro,None,79,WA
10287,20419,222,7723 Firestone Drive,None,79,WA
10288,20550,255,7469 Paradise Ct.,None,79,WA


In [25]:
dimensionCustomer = tablaCustomer.merge(tablaPerson, left_on='CustomerID', right_on='BusinessEntityID')
dimensionCustomer = dimensionCustomer.merge(tablaEmployee, on='BusinessEntityID')

dimensionCustomer = dimensionCustomer.merge(tablaPersonPhone, on='BusinessEntityID')
dimensionCustomer = dimensionCustomer.merge(tablaEmailAddress, on='BusinessEntityID')
dimensionCustomer = dimensionCustomer.merge(customer, on='BusinessEntityID')


dimensionCustomer

,CustomerID,AccountNumber,BusinessEntityID,NameStyle,Title,FirstName,MiddleName,LastName,Suffix,BirthDate,MaritalStatus,Gender,PhoneNumber,EmailAddress,AddressID,AddressLine1,AddressLine2,StateProvinceID,StateProvinceCode
0,1,AW00000001,1,False,None,Ken,J,Sánchez,None,1969-01-29,S,M,697-555-0142,ken0@adventure-works.com,249,4350 Minute Dr.,None,79,WA
1,2,AW00000002,2,False,None,Terri,Lee,Duffy,None,1971-08-01,S,F,819-555-0175,terri0@adventure-works.com,293,7559 Worth Ct.,None,79,WA
2,7,AW00000007,7,False,None,Dylan,A,Miller,None,1987-02-24,M,M,181-555-0156,dylan0@adventure-works.com,49,7048 Laurel,None,79,WA
3,19,AW00000019,19,False,None,Mary,A,Dempsey,None,1978-01-29,S,F,124-555-0114,mary2@adventure-works.com,174,6307 Greenbelt Way,None,79,WA
4,20,AW00000020,20,False,None,Wanida,M,Benshoof,None,1975-03-17,M,F,708-555-0141,wanida0@adventure-works.com,191,6951 Harmony Way,None,79,WA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,214,AW00000214,214,False,None,Andreas,T,Berglund,None,1989-03-28,M,M,181-555-0124,andreas0@adventure-works.com,292,1803 Olive Hill,None,79,WA
286,232,AW00000232,232,False,None,Pat,H,Coleman,None,1970-12-03,S,M,720-555-0158,pat0@adventure-works.com,257,2425 Notre Dame Ave,None,79,WA
287,250,AW00000250,250,False,None,Sheela,H,Word,None,1978-02-10,S,F,210-555-0193,sheela0@adventure-works.com,32509,535 Greendell Pl,None,79,WA
288,268,AW00000268,268,False,None,Ramesh,V,Meyyappan,None,1988-03-13,S,M,182-555-0134,ramesh0@adventure-works.com,273,3848 East 39th Street,None,79,WA


In [26]:
# AGREGAR LAS COLUMNAS NUEVAS

dimensionCustomer["EnglishEducation"] = None
dimensionCustomer["SpanishEducation"] = None
dimensionCustomer["FrenchEducation"] = None
dimensionCustomer["EnglishOccupation"] = None
dimensionCustomer["SpanishOccupation"] = None
dimensionCustomer["FrenchOccupation"] = None
dimensionCustomer["YearlyIncome"] = None
dimensionCustomer["TotalChildren"] = None
dimensionCustomer["NumberChildrenAtHome"] = None
dimensionCustomer["HouseOwnerFlag"] = None
dimensionCustomer["NumberCarsOwned"] = None
dimensionCustomer["DateFirstPurchase"] = None
dimensionCustomer["CommuteDistance"] = None

dimensionCustomer

,CustomerID,AccountNumber,BusinessEntityID,NameStyle,Title,FirstName,MiddleName,LastName,Suffix,BirthDate,...,EnglishOccupation,SpanishOccupation,FrenchOccupation,YearlyIncome,TotalChildren,NumberChildrenAtHome,HouseOwnerFlag,NumberCarsOwned,DateFirstPurchase,CommuteDistance
0,1,AW00000001,1,False,None,Ken,J,Sánchez,None,1969-01-29,...,None,None,None,None,None,None,None,None,None,None
1,2,AW00000002,2,False,None,Terri,Lee,Duffy,None,1971-08-01,...,None,None,None,None,None,None,None,None,None,None
2,7,AW00000007,7,False,None,Dylan,A,Miller,None,1987-02-24,...,None,None,None,None,None,None,None,None,None,None
3,19,AW00000019,19,False,None,Mary,A,Dempsey,None,1978-01-29,...,None,None,None,None,None,None,None,None,None,None
4,20,AW00000020,20,False,None,Wanida,M,Benshoof,None,1975-03-17,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,214,AW00000214,214,False,None,Andreas,T,Berglund,None,1989-03-28,...,None,None,None,None,None,None,None,None,None,None
286,232,AW00000232,232,False,None,Pat,H,Coleman,None,1970-12-03,...,None,None,None,None,None,None,None,None,None,None
287,250,AW00000250,250,False,None,Sheela,H,Word,None,1978-02-10,...,None,None,None,None,None,None,None,None,None,None
288,268,AW00000268,268,False,None,Ramesh,V,Meyyappan,None,1988-03-13,...,None,None,None,None,None,None,None,None,None,None


In [27]:
dimensionCustomer.rename(columns={
    "CustomerID":"CustomerKey",
    "AccountNumber":"CustomerAlternateKey",
}, inplace=True)

dimensionCustomer.drop(columns=[
    'BusinessEntityID'
], inplace=True)

dimensionCustomer

,CustomerKey,CustomerAlternateKey,NameStyle,Title,FirstName,MiddleName,LastName,Suffix,BirthDate,MaritalStatus,...,EnglishOccupation,SpanishOccupation,FrenchOccupation,YearlyIncome,TotalChildren,NumberChildrenAtHome,HouseOwnerFlag,NumberCarsOwned,DateFirstPurchase,CommuteDistance
0,1,AW00000001,False,None,Ken,J,Sánchez,None,1969-01-29,S,...,None,None,None,None,None,None,None,None,None,None
1,2,AW00000002,False,None,Terri,Lee,Duffy,None,1971-08-01,S,...,None,None,None,None,None,None,None,None,None,None
2,7,AW00000007,False,None,Dylan,A,Miller,None,1987-02-24,M,...,None,None,None,None,None,None,None,None,None,None
3,19,AW00000019,False,None,Mary,A,Dempsey,None,1978-01-29,S,...,None,None,None,None,None,None,None,None,None,None
4,20,AW00000020,False,None,Wanida,M,Benshoof,None,1975-03-17,M,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,214,AW00000214,False,None,Andreas,T,Berglund,None,1989-03-28,M,...,None,None,None,None,None,None,None,None,None,None
286,232,AW00000232,False,None,Pat,H,Coleman,None,1970-12-03,S,...,None,None,None,None,None,None,None,None,None,None
287,250,AW00000250,False,None,Sheela,H,Word,None,1978-02-10,S,...,None,None,None,None,None,None,None,None,None,None
288,268,AW00000268,False,None,Ramesh,V,Meyyappan,None,1988-03-13,S,...,None,None,None,None,None,None,None,None,None,None


In [28]:
duplicados = dimensionCustomer.duplicated()
print(duplicados)


0      False
1      False
2      False
3      False
4      False
       ...  
285    False
286    False
287    False
288    False
289    False
Length: 290, dtype: bool


CARGAR A LA BODEGA

In [29]:
dimensionCustomer.to_sql('dimensionCustomer',motorBodegaDatos, if_exists='replace',index=False)

22